# Atlas GasNet E3: MLP, Genetic Programming and baseline

Genetic Programming is the accepted second technique (2026-11-22).
Both learned models use identical pre-execution inputs and training-only transforms.
The selector-median baseline fits training labels only. This notebook is separate
from the completed E2 notebook and its retained results.

Supply four complete E3 extraction directories, with manifests and listed Parquet
parts, under `data/extractions/e3-v1`. The published E2 archive cannot replace E3.
Current campaign: 155,435 train / 39,111 validation / 59,504 temporal-test /
145,124 USDT rows. These counts are not model performance findings.

Run setup and preparation first. Fitting starts only when its cell is executed;
full-population CPU genetic programming can take substantial time. Review
validation diagnostics, then explicitly freeze models before running final tests.


In [ ]:
from pathlib import Path
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
ROOT = Path("/content/atlas-gasnet") if IN_COLAB else Path.cwd()
if not (ROOT / "training/pipeline.py").is_file():
    if not IN_COLAB:
        raise RuntimeError("Start Jupyter from the Atlas GasNet repository root.")
    subprocess.run([
        "git", "clone", "--branch", "work",
        "https://github.com/brianUtn98/atlas-gasnet.git", str(ROOT)
    ], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-r",
    str(ROOT / "training/requirements-symbolic.txt")
], check=True)
print("Code revision:", subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip())
print("Use the revision containing the E3 integration; retain this revision with results.")

def run_python(script):
    result = subprocess.run(
        [sys.executable, "-c", script], cwd=ROOT, capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
    result.check_returncode()


In [ ]:
run_python("""
from pathlib import Path
from training.bundles import prepare, load_prepared
from training.prepare_dataset import read_json

source = Path("data/extractions/e3-v1")
prepared = Path("models/e3-prepared")
if not source.is_dir():
    raise FileNotFoundError("Supply the four complete E3 extraction directories first.")
if prepared.exists():
    load_prepared(prepared)
    if read_json(prepared / "provenance.json")["mode"] != "e3":
        raise ValueError("Prepared inputs must belong to E3.")
else:
    prepare(source, prepared, "e3")
provenance = read_json(prepared / "provenance.json")
print({split: data["rows"] for split, data in provenance["splits"].items()})
print("Preprocessing fitted on train only. No models trained.")
""")


## Fit candidates, select on validation

The checked-in candidate list uses seed 42, 500 individuals and 20 generations,
with parsimony coefficients .001 and .01. These are starting choices, not tuned
findings. Each candidate fits full train; lowest validation MAE wins, with first
candidate breaking ties. MLP uses its existing best-validation-MSE early stopping.
No final-test rows enter fitting or selection. Use a fresh run directory if changing
candidate configurations or features. Existing runs are verified and retained.


In [ ]:
run_python("""
from pathlib import Path
from training.bundles import train_bundle, load_run
from training.prepare_dataset import read_json
from training.symbolic_experiment import configurations

run = Path("models/e3-run")
if run.exists():
    *_, manifest = load_run(run)
    if "symbolic" not in manifest.get("estimators", []):
        raise ValueError("Existing run does not include the symbolic estimator.")
    print("Existing complete comparison run retained.")
else:
    candidates = configurations(read_json(Path("config/training.e3.symbolic.json")))
    train_bundle(Path("models/e3-prepared"), run, symbolic_configs=candidates)
print("MLP, symbolic and baseline ready; final tests remain closed.")
""")


In [ ]:
run_python("""
from pathlib import Path
from training.evaluation import evaluate
from training.prepare_dataset import read_json

run = Path("models/e3-run")
report = evaluate(run, Path("models/e3-prepared"),
                  validation_only=True, verification=(run / "validation").exists())
report = read_json(run / "validation/metrics.json")
for name, summary in report["validation_diagnostics"]["estimators"].items():
    print(name, summary["aggregate"])
history = read_json(run / "symbolic-history.json")
print("Selected symbolic candidate:", history["selected_candidate"])
for candidate in history["candidates"]:
    print(candidate["candidate"], candidate["configuration"],
          "validation MAE:", candidate["validation_mae_gas"])
print("Display-only gas formula:", read_json(run / "symbolic.json")["expression_display_only"])
print("Formula inputs are ordered transformed features; use predict for inference.")
""")


## Freeze models before opening final tests

Finish feature and configuration choices using train and validation only. Change
the following switch only once both models are frozen. A new tuning cycle after
seeing test scores requires a different, unconsulted test window.


In [ ]:
MODELS_FROZEN = False


In [ ]:
if not MODELS_FROZEN:
    raise RuntimeError("Review validation and freeze both models before opening final tests.")
run_python("""
from pathlib import Path
from training.evaluation import evaluate
from training.prepare_dataset import read_json

run = Path("models/e3-run")
evaluate(run, Path("models/e3-prepared"), verification=(run / "evaluation").exists())
report = read_json(run / "evaluation/metrics.json")
for split, section in report["final_tests"].items():
    print(split)
    for name, summary in section["estimators"].items():
        print(name, summary["aggregate"])
""")


In [ ]:
if not MODELS_FROZEN:
    raise RuntimeError("Final results are available only after model freeze and evaluation.")
import json
from IPython.display import Image, display

run = ROOT / "models/e3-run"
history = json.loads((run / "history.json").read_text())
print("MLP selected epoch:", history["best_epoch"])
for split in ("test", "unseen-contract"):
    display(Image(filename=str(run / f"evaluation/{split}.png")))
print("One week and one unseen contract limit generalization claims.")

if IN_COLAB:
    import shutil
    from google.colab import files

    archive = shutil.make_archive("/content/atlas-gasnet-e3-results", "zip", run)
    files.download(archive)
